# 13 — Feature Engineering, Data Leakage, and Dataset Design

This notebook practices train-only preprocessing, missingness flags, one-hot encoding, interaction features, and leakage checks.

In [ ]:
import numpy as np

## 1. Create a Mixed-Type Dataset

In [ ]:
rng = np.random.default_rng(42)

n = 260
age = rng.normal(35, 9, size=n)
income = rng.lognormal(mean=8.2, sigma=0.45, size=n)
transactions_30d = rng.poisson(8, size=n).astype(float)

income[rng.choice(n, size=30, replace=False)] = np.nan
transactions_30d[rng.choice(n, size=20, replace=False)] = np.nan

device = rng.choice(['ios', 'android', 'web'], size=n, p=[0.35, 0.45, 0.20])

def sigmoid(z):
    return 1 / (1 + np.exp(-np.clip(z, -40, 40)))

income_signal = np.nan_to_num(np.log1p(income), nan=np.nanmedian(np.log1p(income)))
tx_signal = np.nan_to_num(transactions_30d, nan=np.nanmedian(transactions_30d))
logit = -5.5 + 0.05 * age + 0.35 * income_signal - 0.10 * tx_signal + 0.55 * (device == 'web')
prob = sigmoid(logit)
y = rng.binomial(1, prob)
X_num = np.column_stack([age, income, transactions_30d])

X_num.shape, device[:5], y.mean()

## 2. Split First

In [ ]:
def train_test_split_numpy(X_num, categories, y, test_size=0.25, seed=42):
    rng = np.random.default_rng(seed)
    n = len(y)
    idx = rng.permutation(n)
    test_n = int(n * test_size)
    test_idx = idx[:test_n]
    train_idx = idx[test_n:]
    return X_num[train_idx], X_num[test_idx], categories[train_idx], categories[test_idx], y[train_idx], y[test_idx]

X_train_num, X_test_num, cat_train, cat_test, y_train, y_test = train_test_split_numpy(X_num, device, y)

X_train_num.shape, X_test_num.shape

## 3. Train-Only Imputation and Scaling

In [ ]:
def fit_median_imputer(X_train):
    return np.nanmedian(X_train, axis=0)


def transform_median_imputer(X, medians):
    missing_flags = np.isnan(X).astype(float)
    X_filled = np.where(np.isnan(X), medians, X)
    return X_filled, missing_flags


def fit_standardizer(X_train):
    mean = X_train.mean(axis=0)
    std = X_train.std(axis=0)
    std = np.where(std == 0, 1, std)
    return mean, std


def transform_standardizer(X, mean, std):
    return (X - mean) / std

medians = fit_median_imputer(X_train_num)
X_train_filled, train_missing_flags = transform_median_imputer(X_train_num, medians)
X_test_filled, test_missing_flags = transform_median_imputer(X_test_num, medians)

mean, std = fit_standardizer(X_train_filled)
X_train_scaled = transform_standardizer(X_train_filled, mean, std)
X_test_scaled = transform_standardizer(X_test_filled, mean, std)

np.round(mean, 3), np.round(std, 3)

## 4. One-Hot Encoding with Unknown Handling

In [ ]:
def fit_one_hot(categories):
    return {cat: i for i, cat in enumerate(sorted(set(categories)))}


def transform_one_hot(categories, mapping):
    X = np.zeros((len(categories), len(mapping)))
    for row, cat in enumerate(categories):
        if cat in mapping:
            X[row, mapping[cat]] = 1
    return X

mapping = fit_one_hot(cat_train)
X_train_cat = transform_one_hot(cat_train, mapping)
X_test_cat = transform_one_hot(cat_test, mapping)

mapping, X_train_cat[:5]

## 5. Build Final Feature Matrix

In [ ]:
train_interaction = X_train_scaled[:, [0]] * X_train_scaled[:, [2]]
test_interaction = X_test_scaled[:, [0]] * X_test_scaled[:, [2]]

X_train_final = np.column_stack([X_train_scaled, train_missing_flags, X_train_cat, train_interaction])
X_test_final = np.column_stack([X_test_scaled, test_missing_flags, X_test_cat, test_interaction])

X_train_final.shape, X_test_final.shape

## 6. Simple Logistic Regression

In [ ]:
def train_logistic_regression(X, y, lr=0.1, steps=1200):
    X_bias = np.column_stack([np.ones(X.shape[0]), X])
    beta = np.zeros(X_bias.shape[1])
    for _ in range(steps):
        p = sigmoid(X_bias @ beta)
        gradient = (1 / len(y)) * X_bias.T @ (p - y)
        beta -= lr * gradient
    return beta


def predict(X, beta, threshold=0.5):
    X_bias = np.column_stack([np.ones(X.shape[0]), X])
    p = sigmoid(X_bias @ beta)
    return (p >= threshold).astype(int)

beta = train_logistic_regression(X_train_final, y_train)
pred = predict(X_test_final, beta)
np.mean(pred == y_test)

## 7. Leakage Reflection

Features must be available at prediction time. Preprocessing must be fitted on training data only. Target-derived transformations must happen inside cross-validation.